# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedsufian10/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type: Ranking / Scoring.**
While we are predicting a binary label (will it decline or not?), the ultimate goal is not classification.
The business action requires a prioritized queue of pages for editors to review. Therefore,
this is a ranking task: we need to score the expected impact and rank the pages from highest priority to lowest.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**The Target (Proxy):** `is_declining_label` (which is derived from `trend_direction == 'down'`).
This is a **proxy label**. Our true desired outcome is "did the page recover after an editor refreshed it?" but we don't have causal experimental data for that yet. Instead, we use a proxy: "is the page currently exhibiting a sustained decline in traffic?" By targeting pages that are actively declining, we aim to protect and recover at-risk traffic.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success Metric: Precision@K** (e.g., Precision@50).
Accuracy is not a good metric here because most pages are not declining, and we don't care about the bottom of the list. If an editor team only has the capacity to rewrite 50 pages this week, the only number that matters is: *of the top 50 pages our model recommended, how many were actually declining?*

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
import pandas as pd
import os
import sys, subprocess

# 1. Colab Setup (downloads the data if it's not there)
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks" or os.path.basename(os.getcwd()) == "work":
    os.chdir("..")

# 2. Safely load the data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 3. Define our proxy target column
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# 4. Show the unit of analysis
print(f"Unit of analysis: One row = one pseudonymized content item (page) aggregated over a 90-day window.")
print(f"Total Rows (Content Items): {df.shape[0]:,}")
print("\nHere is a slice of our dataframe showing the unit and our target column:")
df[["content_id", "content_type", "impressions_90d", "trend_direction", "is_declining_label"]].head()

Unit of analysis: One row = one pseudonymized content item (page) aggregated over a 90-day window.
Total Rows (Content Items): 30,000

Here is a slice of our dataframe showing the unit and our target column:


,content_id,content_type,impressions_90d,trend_direction,is_declining_label
0,content_304f48230142,keyword article,3803,down,1
1,content_a1fb4e703a9e,keyword article,15320,down,1
2,content_9aa793d4d895,keyword article,12581,down,1
3,content_331d6c4de07b,keyword article,11751,stable,0
4,content_d99b7a2d90ca,keyword article,19140,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed, hand-written rule (like `stale AND visible`) is often very accurate at the very top of the list, but it quickly runs out of signal deeper in the inventory. A machine learning model can untangle complex, shifting relationships between multiple metrics (like CTR, average position, age, and search volume) to find non-obvious refresh opportunities that a simple `if-statement` would miss.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.